# Multi-asset portfolio valuation at scale

**Prerequisites:** individual instrument notebooks in **02_pricing/instruments/** and the portfolio construction notebook in this directory.

This notebook builds a **representative multi-asset portfolio** spanning seventeen instrument types: **bonds** (fixed and floating), **term loans**, **revolving credit facilities**, **CDS**, **CDS indices**, **CDS tranches**, **CDS options**, **CLOs**, **ABS**, **interest rate swaps**, **swaptions**, **IR futures**, **spot equity**, **variance swaps**, **FX swaps**, and **convertible bonds**.

After valuation it extracts the **risk metrics** an investment group cares about most: **bucketed DV01**, **bucketed CS01**, **delta**, **gamma**, **vega**, **theta**, and their portfolio-level aggregations. It then replicates the book to **500** and **3,000** positions and measures valuation throughput.

In [1]:
import copy
import json
import time
from datetime import date

from finstack.core.market_data import DiscountCurve, ForwardCurve, HazardCurve, MarketContext
from finstack.portfolio import aggregate_metrics, value_portfolio
from finstack.scenarios import apply_scenario_to_market, build_scenario_spec
from finstack.valuations import attribute_pnl, validate_instrument_json

AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

print("Imports OK.")

Imports OK.


## 1. Market data

A shared `MarketContext` provides everything the seventeen instrument types need:

| Data | ID | Used by |
|------|----|---------|
| USD OIS discount | `USD-OIS` | all USD instruments |
| EUR OIS discount | `EUR-OIS` | FX swaps (foreign leg) |
| USD SOFR 3M forward | `USD-SOFR-3M` | FRN, IRS float, revolver, swaption, IR future |
| USD IG risky discount | `USD-IG` | convertible bonds (discount) |
| USD Credit BBB discount | `USD-CREDIT-BBB` | convertible bonds (credit) |
| Corporate hazard | `CORP-HAZARD` | CDS, CDS option |
| Index hazard | `CDX-HAZ` | CDS tranche, CDS index |
| Base correlation | `CDX-BC` | CDS tranche |
| CDS spread vol surface | `CDS-SPREAD-VOL` | CDS option |
| Swaption vol surface | `USD-SWPNVOL` | swaptions |
| Equity vol surface | `TECH-VOL` | convertible bond |
| Equity spot / div prices | `TECH`, `AAPL-SPOT`, etc. | equity, convertible, variance swap |
| FX quotes | `EUR/USD` | FX swaps |

In [2]:
mc = MarketContext()

mc.insert(DiscountCurve(
    "USD-OIS", AS_OF,
    [(0.0, 1.0), (0.25, 0.9888), (0.5, 0.9775), (1.0, 0.955), (2.0, 0.91),
     (3.0, 0.87), (5.0, 0.80), (7.0, 0.73), (10.0, 0.65)],
    day_count="act_365f",
))

mc.insert(DiscountCurve(
    "EUR-OIS", AS_OF,
    [(0.0, 1.0), (0.25, 0.9925), (0.5, 0.985), (1.0, 0.97), (2.0, 0.94),
     (3.0, 0.91), (5.0, 0.85), (7.0, 0.79), (10.0, 0.70)],
    day_count="act_365f",
))

mc.insert(ForwardCurve(
    "USD-SOFR-3M", 0.25,
    [(0.0, 0.045), (0.5, 0.046), (1.0, 0.047), (2.0, 0.048),
     (3.0, 0.049), (5.0, 0.050), (10.0, 0.052)],
    base_date=AS_OF,
    day_count="act_360",
))

mc.insert(DiscountCurve(
    "USD-IG", AS_OF,
    [(0.0, 1.0), (1.0, 0.945), (3.0, 0.84), (5.0, 0.75), (10.0, 0.58)],
    day_count="act_365f",
))

mc.insert(DiscountCurve(
    "USD-CREDIT-BBB", AS_OF,
    [(0.0, 1.0), (1.0, 0.935), (3.0, 0.82), (5.0, 0.72), (10.0, 0.52)],
    day_count="act_365f",
))

mc.insert(HazardCurve(
    "CORP-HAZARD", AS_OF,
    [(0.5, 0.018), (1.0, 0.020), (3.0, 0.024), (5.0, 0.028), (10.0, 0.032)],
    recovery_rate=0.40,
))

mc.insert(HazardCurve(
    "CDX-HAZ", AS_OF,
    [(1.0, 0.01), (3.0, 0.015), (5.0, 0.02), (10.0, 0.025)],
    recovery_rate=0.40,
))

state = json.loads(mc.to_json())

state["curves"].append({
    "type": "base_correlation",
    "id": "CDX-BC",
    "detachment_points": [3.0, 7.0, 10.0, 15.0, 30.0, 100.0],
    "correlations": [0.18, 0.28, 0.35, 0.45, 0.65, 0.85],
})

state["credit_indices"] = [{
    "id": "CDX.NA.IG.HAZARD",
    "num_constituents": 125,
    "recovery_rate": 0.4,
    "index_credit_curve_id": "CDX-HAZ",
    "base_correlation_curve_id": "CDX-BC",
}]

state["surfaces"] = [
    {
        "id": "CDS-SPREAD-VOL",
        "expiries": [0.25, 0.5, 1.0, 2.0],
        "strikes": [50.0, 100.0, 150.0, 200.0, 300.0],
        "secondary_axis": "strike",
        "interpolation_mode": "vol",
        "vols_row_major": [0.30] * 20,
    },
    {
        "id": "TECH-VOL",
        "expiries": [0.25, 0.5, 1.0, 2.0, 5.0],
        "strikes": [20.0, 30.0, 40.0, 50.0, 60.0],
        "secondary_axis": "strike",
        "interpolation_mode": "vol",
        "vols_row_major": [0.35] * 25,
    },
    {
        "id": "USD-SWPNVOL",
        "expiries": [0.25, 0.5, 1.0, 2.0, 5.0],
        "strikes": [0.02, 0.03, 0.04, 0.05, 0.06],
        "secondary_axis": "strike",
        "interpolation_mode": "vol",
        "vols_row_major": [0.20] * 25,
    },
]

state.setdefault("prices", {})
state["prices"]["TECH"] = {"price": {"amount": 42.0, "currency": "USD"}}
state["prices"]["AAPL-SPOT"] = {"price": {"amount": 185.0, "currency": "USD"}}
state["prices"]["AAPL-DIV"] = {"unitless": 0.005}
state["prices"]["SPX-SPOT"] = {"price": {"amount": 5200.0, "currency": "USD"}}
state["prices"]["SPX-DIV"] = {"unitless": 0.015}

state["fx"] = {
    "config": {"pivot_currency": "USD", "enable_triangulation": False, "cache_capacity": 256},
    "quotes": [["EUR", "USD", 1.08]],
}

MARKET_JSON = json.dumps(state)

curve_ids = [c.get("id") for c in state["curves"] if isinstance(c, dict)]
print(f"Market snapshot ready \u2014 {len(curve_ids)} curves, {len(state['surfaces'])} surfaces, {len(state['prices'])} prices")
print("Curve IDs:", curve_ids)

Market snapshot ready — 8 curves, 3 surfaces, 5 prices
Curve IDs: ['CDX-HAZ', 'CORP-HAZARD', 'EUR-OIS', 'USD-CREDIT-BBB', 'USD-IG', 'USD-OIS', 'USD-SOFR-3M', 'CDX-BC']


## 2. Base instrument templates

Each helper returns a `(instrument_id, instrument_spec)` tuple. The `instrument_spec` is the tagged `{"type": ..., "spec": ...}` dict ready to embed in a portfolio position.

In [3]:
def fixed_bond(idx: int) -> tuple[str, dict]:
    iid = f"BOND-FIXED-{idx}"
    coupon = 0.04 + 0.005 * (idx % 5)
    mat_year = 2028 + (idx % 8)
    return iid, {"type": "bond", "spec": {
        "id": iid, "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15", "maturity": f"{mat_year}-01-15",
        "discount_curve_id": "USD-OIS", "accrual_method": "Linear",
        "settlement_days": 1, "ex_coupon_days": 0,
        "cashflow_spec": {"Fixed": {
            "coupon_type": "Cash", "freq": {"count": 6, "unit": "months"},
            "dc": "Thirty360", "bdc": "following", "calendar_id": "weekends_only",
            "end_of_month": False, "payment_lag_days": 0, "rate": str(coupon), "stub": "None",
        }},
        "call_put": None, "credit_curve_id": None,
        "attributes": {"tags": ["fixed-income"], "meta": {"sector": "IG"}},
        "pricing_overrides": {},
    }}


def floating_bond(idx: int) -> tuple[str, dict]:
    iid = f"BOND-FRN-{idx}"
    spread = 100 + 25 * (idx % 6)
    mat_year = 2027 + (idx % 5)
    return iid, {"type": "bond", "spec": {
        "id": iid, "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15", "maturity": f"{mat_year}-01-15",
        "discount_curve_id": "USD-OIS", "accrual_method": "Linear",
        "settlement_days": 2, "ex_coupon_days": 0,
        "cashflow_spec": {"Floating": {
            "coupon_type": "Cash", "freq": {"count": 3, "unit": "months"}, "stub": "ShortFront",
            "rate_spec": {
                "index_id": "USD-SOFR-3M", "spread_bp": str(spread), "gearing": "1",
                "gearing_includes_spread": True, "floor_bp": "0",
                "all_in_floor_bp": None, "cap_bp": None, "index_cap_bp": None,
                "dc": "Act360", "bdc": "modified_following", "calendar_id": "weekends_only",
                "fixing_calendar_id": None, "reset_freq": {"count": 3, "unit": "months"},
                "reset_lag_days": 2, "payment_lag_days": 0, "end_of_month": False,
            },
        }},
        "call_put": None, "credit_curve_id": None,
        "attributes": {"tags": ["fixed-income", "frn"], "meta": {}}, "pricing_overrides": {},
    }}


def term_loan(idx: int) -> tuple[str, dict]:
    iid = f"TL-{idx}"
    rate_bp = 500 + 50 * (idx % 4)
    mat_year = 2028 + (idx % 4)
    return iid, {"type": "term_loan", "spec": {
        "id": iid, "notional_limit": {"amount": "10000000", "currency": "USD"},
        "currency": "USD", "issue_date": "2024-01-01", "maturity": f"{mat_year}-01-01",
        "discount_curve_id": "USD-OIS", "rate": {"Fixed": {"rate_bp": rate_bp}},
        "day_count": "Act360", "frequency": {"count": 3, "unit": "months"},
        "bdc": "modified_following", "calendar_id": None, "stub": "None",
        "amortization": {"PercentPerPeriod": {"bp": 250}}, "coupon_type": "Cash",
        "settlement_days": 2, "attributes": {"tags": ["leveraged-loan"], "meta": {}},
        "pricing_overrides": {},
    }}


def revolver(idx: int) -> tuple[str, dict]:
    iid = f"RCF-{idx}"
    spread = 200 + 50 * (idx % 4)
    mat_year = 2027 + (idx % 3)
    return iid, {"type": "revolving_credit", "spec": {
        "id": iid, "commitment_amount": {"amount": "50000000", "currency": "USD"},
        "drawn_amount": {"amount": "10000000", "currency": "USD"},
        "commitment_date": "2024-01-01", "maturity": f"{mat_year}-01-01",
        "discount_curve_id": "USD-OIS", "day_count": "Act360",
        "frequency": {"count": 3, "unit": "months"}, "stub": "ShortFront", "recovery_rate": 0.70,
        "base_rate_spec": {"Floating": {
            "index_id": "USD-SOFR-3M", "spread_bp": str(spread), "gearing": "1",
            "gearing_includes_spread": True, "floor_bp": "0",
            "all_in_floor_bp": None, "cap_bp": None, "index_cap_bp": None,
            "dc": "Act360", "bdc": "modified_following", "calendar_id": "weekends_only",
            "fixing_calendar_id": None, "reset_freq": {"count": 3, "unit": "months"},
            "reset_lag_days": 2, "payment_lag_days": 0, "end_of_month": False,
        }},
        "draw_repay_spec": {"Deterministic": [
            {"date": "2024-06-01", "amount": {"amount": "5000000", "currency": "USD"}, "is_draw": True},
            {"date": "2025-06-01", "amount": {"amount": "3000000", "currency": "USD"}, "is_draw": False},
        ]},
        "fees": {"commitment_fee_tiers": [{"threshold": "0", "bps": "25"}],
                 "usage_fee_tiers": [{"threshold": "0", "bps": "10"}], "facility_fee_bp": 5.0},
        "attributes": {"tags": ["revolving"], "meta": {}}, "pricing_overrides": {},
    }}


def cds(idx: int) -> tuple[str, dict]:
    iid = f"CDS-{idx}"
    spread = 80 + 20 * (idx % 5)
    side = "pay" if idx % 2 == 0 else "receive"
    mat_year = 2028 + (idx % 5)
    return iid, {"type": "credit_default_swap", "spec": {
        "id": iid, "notional": {"amount": 10_000_000.0, "currency": "USD"},
        "side": side, "convention": "isda_na",
        "premium": {"start": "2025-03-20", "end": f"{mat_year}-03-20",
                    "frequency": {"count": 3, "unit": "months"}, "stub": "ShortFront",
                    "bdc": "following", "calendar_id": "usny", "day_count": "Act360",
                    "spread_bp": float(spread), "discount_curve_id": "USD-OIS"},
        "protection": {"credit_curve_id": "CORP-HAZARD", "recovery_rate": 0.4, "settlement_delay": 3},
        "pricing_overrides": {}, "attributes": {"tags": ["credit"], "meta": {}},
    }}


def cds_index(idx: int) -> tuple[str, dict]:
    iid = f"CDX-IDX-{idx}"
    spread = 50 + 10 * (idx % 5)
    side = "pay" if idx % 2 == 0 else "receive"
    mat_year = 2029 + (idx % 3)
    return iid, {"type": "cds_index", "spec": {
        "id": iid, "index_name": "CDX.NA.IG", "series": 42, "version": 1,
        "notional": {"amount": "10000000", "currency": "USD"}, "index_factor": 1.0,
        "side": side, "convention": "isda_na",
        "premium": {"start": "2025-03-20", "end": f"{mat_year}-12-20",
                    "frequency": {"count": 3, "unit": "months"}, "stub": "ShortFront",
                    "bdc": "following", "calendar_id": None, "day_count": "Act360",
                    "spread_bp": float(spread), "discount_curve_id": "USD-OIS"},
        "protection": {"credit_curve_id": "CDX-HAZ", "recovery_rate": 0.4, "settlement_delay": 3},
        "pricing": "SingleCurve", "constituents": [],
        "pricing_overrides": {}, "attributes": {"tags": ["credit-index"], "meta": {}},
    }}


def cds_tranche(idx: int) -> tuple[str, dict]:
    iid = f"CDX-TR-{idx}"
    attach_detach = [(0.0, 3.0), (3.0, 7.0), (7.0, 10.0)]
    a, d = attach_detach[idx % 3]
    coupon = 500.0 if a == 0.0 else 100.0
    return iid, {"type": "cds_tranche", "spec": {
        "id": iid, "index_name": "CDX.NA.IG", "series": 42,
        "attach_pct": a, "detach_pct": d,
        "notional": {"amount": "10000000", "currency": "USD"}, "maturity": "2029-12-20",
        "running_coupon_bp": coupon, "frequency": {"count": 3, "unit": "months"},
        "day_count": "Act360", "bdc": "following", "calendar_id": "weekends_only",
        "discount_curve_id": "USD-OIS", "credit_index_id": "CDX.NA.IG.HAZARD",
        "side": "buy_protection", "accumulated_loss": 0.0, "standard_imm_dates": False,
        "attributes": {"tags": ["structured-credit"], "meta": {}},
    }}


def cds_option(idx: int) -> tuple[str, dict]:
    iid = f"CDSOPT-{idx}"
    strike = 0.008 + 0.002 * (idx % 4)
    opt_type = "call" if idx % 2 == 0 else "put"
    return iid, {"type": "cds_option", "spec": {
        "id": iid, "strike": str(strike), "option_type": opt_type,
        "exercise_style": "european", "expiry": "2025-06-20", "cds_maturity": "2030-06-20",
        "day_count": "Act360", "notional": {"amount": "10000000", "currency": "USD"},
        "settlement": "cash", "recovery_rate": 0.4,
        "discount_curve_id": "USD-OIS", "credit_curve_id": "CORP-HAZARD",
        "vol_surface_id": "CDS-SPREAD-VOL",
        "underlying_is_index": False, "index_factor": None, "forward_spread_adjust": "0",
        "pricing_overrides": {}, "attributes": {"tags": ["credit-vol"], "meta": {}},
    }}


def _pool_assets(iid: str, n: int = 5) -> list[dict]:
    return [{
        "id": f"{iid}-LOAN-{j}",
        "asset_type": {"type": "FirstLienLoan", "industry": None},
        "balance": {"amount": "2000000", "currency": "USD"},
        "rate": 0.055 + 0.005 * (j % 3), "spread_bps": 300.0 + 50.0 * (j % 4),
        "index_id": None, "maturity": f"{2029 + (j % 3)}-01-01",
        "credit_quality": "BB", "industry": "Technology", "obligor_id": f"OBL-{j}",
        "is_defaulted": False, "recovery_amount": None, "purchase_price": None,
        "acquisition_date": None, "day_count": "Act360",
        "smm_override": None, "mdr_override": None,
    } for j in range(n)]


def _structured_credit_spec(iid: str, deal_type: str, idx: int) -> dict:
    return {"type": "structured_credit", "spec": {
        "id": iid, "deal_type": deal_type,
        "pool": {
            "id": f"{iid}-POOL", "deal_type": deal_type, "assets": _pool_assets(iid),
            "cumulative_defaults": {"amount": "0", "currency": "USD"},
            "cumulative_recoveries": {"amount": "0", "currency": "USD"},
            "cumulative_prepayments": {"amount": "0", "currency": "USD"},
            "reinvestment_period": None,
            "collection_account": {"amount": "0", "currency": "USD"},
            "reserve_account": {"amount": "0", "currency": "USD"},
            "excess_spread_account": {"amount": "0", "currency": "USD"},
        },
        "tranches": {"tranches": [{
            "id": f"{iid}-A", "attachment_point": 0.0, "detachment_point": 100.0,
            "behavior_type": "Standard", "seniority": "Senior", "rating": None,
            "original_balance": {"amount": "10000000", "currency": "USD"},
            "current_balance": {"amount": "10000000", "currency": "USD"}, "target_balance": None,
            "coupon": {"Fixed": {"rate": 0.05 + 0.005 * (idx % 4)}},
            "oc_trigger": None, "ic_trigger": None,
            "credit_enhancement": {
                "subordination": {"amount": "0", "currency": "USD"},
                "overcollateralization": {"amount": "0", "currency": "USD"},
                "reserve_account": {"amount": "0", "currency": "USD"},
                "excess_spread": 0.0, "cash_trap_active": False,
            },
            "frequency": {"count": 3, "unit": "months"}, "day_count": "Act360",
            "deferred_interest": {"amount": "0", "currency": "USD"},
            "is_revolving": False, "can_reinvest": False,
            "maturity": "2034-01-01", "expected_maturity": None, "payment_priority": 1,
            "attributes": {"tags": [], "meta": {}},
        }], "total_size": {"amount": "10000000", "currency": "USD"}},
        "closing_date": "2024-01-01", "first_payment_date": "2025-04-01",
        "reinvestment_end_date": None, "maturity": "2034-01-01",
        "frequency": {"count": 3, "unit": "months"}, "discount_curve_id": "USD-OIS",
        "payment_calendar_id": "nyse",
        "attributes": {"tags": [deal_type.lower()], "meta": {}},
        "prepayment_spec": {"cpr": 0.15, "curve": None},
        "default_spec": {"cdr": 0.025, "curve": None},
        "recovery_spec": {"rate": 0.4, "recovery_lag": 18},
        "market_conditions": {"refi_rate": 0.04, "original_rate": None, "hpa": None,
                               "unemployment": None, "seasonal_factor": 1.0, "custom_factors": {}},
        "credit_factors": {"credit_score": None, "dti": None, "ltv": None,
                            "delinquency_days": 0, "unemployment_rate": None, "custom_factors": {}},
        "deal_metadata": {"manager_id": None, "servicer_id": None, "master_servicer_id": None,
                           "special_servicer_id": None, "trustee_id": None},
        "behavior_overrides": {"cpr_annual": None, "abs_speed": None, "psa_speed_multiplier": None,
                                "cdr_annual": None, "sda_speed_multiplier": None, "recovery_rate": None,
                                "recovery_lag_months": None, "reinvestment_price": None},
        "default_assumptions": {"base_cdr_annual": 0.02, "base_recovery_rate": 0.4, "base_cpr_annual": 0.15,
                                 "psa_speed": None, "sda_speed": None, "abs_speed_monthly": None,
                                 "cpr_by_asset_type": {}, "cdr_by_asset_type": {}, "recovery_by_asset_type": {}},
    }}


def clo_deal(idx: int) -> tuple[str, dict]:
    iid = f"CLO-{idx}"
    return iid, _structured_credit_spec(iid, "CLO", idx)


def abs_deal(idx: int) -> tuple[str, dict]:
    iid = f"ABS-{idx}"
    return iid, _structured_credit_spec(iid, "ABS", idx)


def irs(idx: int) -> tuple[str, dict]:
    iid = f"IRS-{idx}"
    rate = 0.040 + 0.005 * (idx % 4)
    side = "pay" if idx % 2 == 0 else "receive"
    mat_year = 2028 + (idx % 7)
    return iid, {"type": "interest_rate_swap", "spec": {
        "id": iid, "notional": {"amount": 10_000_000.0, "currency": "USD"}, "side": side,
        "fixed": {"discount_curve_id": "USD-OIS", "rate": rate,
                  "frequency": {"count": 6, "unit": "months"}, "day_count": "Thirty360",
                  "bdc": "modified_following", "calendar_id": None, "stub": "None",
                  "start": "2025-04-15", "end": f"{mat_year}-04-15",
                  "par_method": None, "compounding_simple": True},
        "float": {"discount_curve_id": "USD-OIS", "forward_curve_id": "USD-SOFR-3M",
                  "spread_bp": 0.0, "frequency": {"count": 3, "unit": "months"},
                  "day_count": "Act360", "bdc": "modified_following", "calendar_id": None,
                  "stub": "None", "reset_lag_days": 2,
                  "start": "2025-04-15", "end": f"{mat_year}-04-15", "compounding": "Simple"},
        "attributes": {"tags": ["rates"], "meta": {}},
    }}


def swaption(idx: int) -> tuple[str, dict]:
    iid = f"SWPN-{idx}"
    strike = 0.035 + 0.005 * (idx % 4)
    opt_type = "call" if idx % 2 == 0 else "put"
    swap_end_year = 2030 + (idx % 5)
    return iid, {"type": "swaption", "spec": {
        "id": iid, "option_type": opt_type,
        "notional": {"amount": "10000000", "currency": "USD"}, "strike": strike,
        "expiry": "2025-07-15", "swap_start": "2025-07-17", "swap_end": f"{swap_end_year}-07-17",
        "fixed_freq": {"count": 6, "unit": "months"}, "float_freq": {"count": 3, "unit": "months"},
        "day_count": "Thirty360", "exercise_style": "european", "settlement": "cash",
        "vol_model": "black", "discount_curve_id": "USD-OIS",
        "forward_curve_id": "USD-SOFR-3M", "vol_surface_id": "USD-SWPNVOL",
        "pricing_overrides": {}, "sabr_params": None,
        "attributes": {"tags": ["rates-vol"], "meta": {}},
    }}


def ir_future(idx: int) -> tuple[str, dict]:
    iid = f"IRF-{idx}"
    expiry_offsets = [(2025, "06"), (2025, "09"), (2025, "12"), (2026, "03"),
                      (2026, "06"), (2026, "09"), (2026, "12"), (2027, "03")]
    end_offsets = [(2025, "09"), (2025, "12"), (2026, "03"), (2026, "06"),
                    (2026, "09"), (2026, "12"), (2027, "03"), (2027, "06")]
    y, m = expiry_offsets[idx % len(expiry_offsets)]
    ey, em = end_offsets[idx % len(end_offsets)]
    price = 95.0 + 0.25 * (idx % 8)
    return iid, {"type": "interest_rate_future", "spec": {
        "id": iid, "notional": {"amount": "1000000", "currency": "USD"},
        "expiry": f"{y}-{m}-17", "fixing_date": f"{y}-{m}-17",
        "period_start": f"{y}-{m}-19",
        "period_end": f"{ey}-{em}-18",
        "quoted_price": price, "day_count": "Act360",
        "position": "long" if idx % 2 == 0 else "short",
        "contract_specs": {"face_value": 1000000.0, "tick_size": 0.0025,
                           "tick_value": 6.25, "delivery_months": 3, "convexity_adjustment": 0.0002},
        "discount_curve_id": "USD-OIS", "forward_curve_id": "USD-SOFR-3M",
        "vol_surface_id": None,
        "attributes": {"tags": ["rates-futures"], "meta": {}},
    }}


def spot_equity(idx: int) -> tuple[str, dict]:
    iid = f"EQ-{idx}"
    return iid, {"type": "equity", "spec": {
        "id": iid, "ticker": "AAPL", "currency": "USD", "shares": 100.0,
        "price_quote": None, "price_id": "AAPL-SPOT", "div_yield_id": "AAPL-DIV",
        "discount_curve_id": "USD-OIS",
        "attributes": {"tags": ["equity"], "meta": {}},
    }}


def variance_swap(idx: int) -> tuple[str, dict]:
    iid = f"VARSWAP-{idx}"
    strike_var = 0.04 + 0.005 * (idx % 4)
    side = "receive" if idx % 2 == 0 else "pay"
    mat_year = 2026 + (idx % 3)
    return iid, {"type": "variance_swap", "spec": {
        "id": iid, "underlying_ticker": "SPX",
        "notional": {"amount": "100000", "currency": "USD"},
        "strike_variance": strike_var, "start_date": AS_OF_STR,
        "maturity": f"{mat_year}-01-15",
        "observation_freq": {"count": 1, "unit": "days"},
        "realized_var_method": "CloseToClose", "side": side,
        "discount_curve_id": "USD-OIS", "day_count": "Act365F",
        "attributes": {"tags": ["equity-vol"], "meta": {}},
    }}


def fx_swap(idx: int) -> tuple[str, dict]:
    iid = f"FXSWAP-{idx}"
    far_year = 2025 + (idx % 2)
    far_rate = 1.085 + 0.005 * (idx % 4)
    return iid, {"type": "fx_swap", "spec": {
        "id": iid, "base_currency": "EUR", "quote_currency": "USD",
        "near_date": "2025-01-17", "far_date": f"{far_year}-07-17",
        "base_notional": {"amount": "1000000", "currency": "EUR"},
        "domestic_discount_curve_id": "USD-OIS", "foreign_discount_curve_id": "EUR-OIS",
        "near_rate": 1.08, "far_rate": far_rate,
        "attributes": {"tags": ["fx"], "meta": {}},
    }}


def convertible_bond(idx: int) -> tuple[str, dict]:
    iid = f"CB-{idx}"
    coupon = 0.015 + 0.005 * (idx % 4)
    ratio = 20.0 + 5.0 * (idx % 3)
    mat_year = 2028 + (idx % 4)
    return iid, {"type": "convertible_bond", "spec": {
        "id": iid, "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15", "maturity": f"{mat_year}-01-15",
        "discount_curve_id": "USD-IG", "credit_curve_id": "USD-CREDIT-BBB",
        "conversion": {"ratio": ratio, "policy": "Voluntary",
                       "anti_dilution": "None", "dividend_adjustment": "None"},
        "underlying_equity_id": "TECH",
        "fixed_coupon": {"coupon_type": "Cash", "rate": coupon,
                         "freq": {"count": 6, "unit": "months"}, "dc": "Thirty360",
                         "bdc": "following", "calendar_id": "weekends_only",
                         "end_of_month": False, "payment_lag_days": 0, "stub": "None"},
        "attributes": {"tags": ["convertible"], "meta": {}}, "pricing_overrides": {},
    }}


def _instrument_desc(instrument_spec: dict) -> str:
    """One-line analyst-readable label for a portfolio position."""
    itype = instrument_spec["type"]
    s = instrument_spec["spec"]
    ref_year = AS_OF.year

    if itype == "bond":
        tenor = int(s["maturity"][:4]) - ref_year
        cf = s["cashflow_spec"]
        if "Fixed" in cf:
            cpn = float(cf["Fixed"]["rate"]) * 100
            return f"Fixed {tenor}Y {cpn:.1f}%"
        sp = cf["Floating"]["rate_spec"]["spread_bp"]
        return f"FRN {tenor}Y SOFR+{sp}bp"

    if itype == "term_loan":
        tenor = int(s["maturity"][:4]) - ref_year
        rate = s["rate"]
        if "Fixed" in rate:
            return f"TL Fixed {tenor}Y {rate['Fixed']['rate_bp']}bp"
        return f"TL Flt {tenor}Y"

    if itype == "revolving_credit":
        tenor = int(s["maturity"][:4]) - ref_year
        drawn = float(s["drawn_amount"]["amount"]) / 1e6
        commit = float(s["commitment_amount"]["amount"]) / 1e6
        return f"Revolver {tenor}Y {drawn:.0f}/{commit:.0f}M"

    if itype == "credit_default_swap":
        tenor = int(s["premium"]["end"][:4]) - ref_year
        spread = float(s["premium"]["spread_bp"])
        side = s["side"][:3].title()
        return f"CDS {side} {tenor}Y {spread:.0f}bp"

    if itype == "cds_index":
        tenor = int(s["premium"]["end"][:4]) - ref_year
        spread = float(s["premium"]["spread_bp"])
        side = s["side"][:3].title()
        return f"CDX IG {side} {tenor}Y {spread:.0f}bp"

    if itype == "cds_tranche":
        a, d = s["attach_pct"], s["detach_pct"]
        return f"CDX Tr {a:.0f}-{d:.0f}%"

    if itype == "cds_option":
        strike_bp = float(s["strike"]) * 10000
        opt = s["option_type"].title()
        return f"CDS Opt {opt} K={strike_bp:.0f}bp"

    if itype == "structured_credit":
        deal = s["deal_type"]
        tr = s["tranches"]["tranches"][0]
        cpn = tr["coupon"]
        if "Fixed" in cpn:
            rate = cpn["Fixed"]["rate"] * 100
            return f"{deal} Snr {rate:.1f}%"
        return f"{deal} Snr"

    if itype == "interest_rate_swap":
        tenor = int(s["fixed"]["end"][:4]) - ref_year
        rate = s["fixed"]["rate"] * 100
        side = s["side"].title()
        return f"IRS {side} {tenor}Y {rate:.1f}%"

    if itype == "swaption":
        swap_tenor = int(s["swap_end"][:4]) - ref_year
        strike = s["strike"] * 100
        opt = s["option_type"].title()
        return f"Swpn {opt} {swap_tenor}Y K={strike:.1f}%"

    if itype == "interest_rate_future":
        exp = s["expiry"][:7]
        pos = s["position"][:1].upper()
        price = s["quoted_price"]
        return f"SOFR Fut {exp} {pos} @{price:.2f}"

    if itype == "equity":
        return f"{s['ticker']} {s['shares']:.0f}sh"

    if itype == "variance_swap":
        tenor = int(s["maturity"][:4]) - ref_year
        side = s["side"][:3].title()
        strike_vol = (s["strike_variance"] ** 0.5) * 100
        return f"VarSwap {s['underlying_ticker']} {side} {tenor}Y σ={strike_vol:.0f}%"

    if itype == "fx_swap":
        return f"FX {s['base_currency']}/{s['quote_currency']} {s['far_date'][:7]}"

    if itype == "convertible_bond":
        tenor = int(s["maturity"][:4]) - ref_year
        cpn = s["fixed_coupon"]["rate"] * 100
        return f"CB {s['underlying_equity_id']} {tenor}Y {cpn:.1f}%"

    return itype


print("Instrument factory functions defined (17 types).")

Instrument factory functions defined (17 types).


## 3. Validate base instruments

Confirm each factory produces valid JSON before embedding in a portfolio.

In [4]:
factories = [
    ("fixed_bond", fixed_bond),
    ("floating_bond", floating_bond),
    ("term_loan", term_loan),
    ("revolver", revolver),
    ("cds", cds),
    ("cds_index", cds_index),
    ("cds_tranche", cds_tranche),
    ("cds_option", cds_option),
    ("clo_deal", clo_deal),
    ("abs_deal", abs_deal),
    ("irs", irs),
    ("swaption", swaption),
    ("ir_future", ir_future),
    ("spot_equity", spot_equity),
    ("variance_swap", variance_swap),
    ("fx_swap", fx_swap),
    ("convertible_bond", convertible_bond),
]

for name, factory in factories:
    iid, spec = factory(0)
    try:
        validate_instrument_json(json.dumps(spec))
        print(f"  {name:20s} -> {iid:20s}  OK")
    except Exception as exc:
        print(f"  {name:20s} -> {iid:20s}  FAIL: {exc}")

  fixed_bond           -> BOND-FIXED-0          OK
  floating_bond        -> BOND-FRN-0            OK
  term_loan            -> TL-0                  OK
  revolver             -> RCF-0                 OK
  cds                  -> CDS-0                 OK
  cds_index            -> CDX-IDX-0             OK
  cds_tranche          -> CDX-TR-0              OK
  cds_option           -> CDSOPT-0              OK
  clo_deal             -> CLO-0                 OK
  abs_deal             -> ABS-0                 OK
  irs                  -> IRS-0                 OK
  swaption             -> SWPN-0                OK
  ir_future            -> IRF-0                 OK
  spot_equity          -> EQ-0                  OK
  variance_swap        -> VARSWAP-0             OK
  fx_swap              -> FXSWAP-0              OK
  convertible_bond     -> CB-0                  OK


## 4. Build the base portfolio (~40 positions)

The base portfolio uses a realistic instrument mix across three fund entities. This validates the full pipeline before replication.

In [5]:
ALLOCATION = [
    (fixed_bond,       0.10),
    (floating_bond,    0.08),
    (term_loan,        0.06),
    (revolver,         0.04),
    (cds,              0.10),
    (cds_index,        0.04),
    (cds_tranche,      0.04),
    (cds_option,       0.04),
    (clo_deal,         0.04),
    (abs_deal,         0.04),
    (irs,              0.10),
    (swaption,         0.05),
    (ir_future,        0.05),
    (spot_equity,      0.05),
    (variance_swap,    0.04),
    (fx_swap,          0.04),
    (convertible_bond, 0.09),
]


def build_positions(target_size: int) -> list[dict]:
    positions = []
    pos_id = 0
    for factory, weight in ALLOCATION:
        count = max(1, int(target_size * weight))
        for i in range(count):
            if pos_id >= target_size:
                break
            iid, spec = factory(i)
            iid_unique = f"{iid}-P{pos_id}"
            spec_copy = copy.deepcopy(spec)
            spec_copy["spec"]["id"] = iid_unique
            _fixup_structured_ids(spec_copy, iid, iid_unique)
            desc = _instrument_desc(spec_copy)
            positions.append({
                "position_id": f"POS-{pos_id} ({desc})",
                "entity_id": f"FUND-{(pos_id % 3) + 1}",
                "instrument_id": iid_unique,
                "instrument_spec": spec_copy,
                "quantity": 1.0,
                "unit": "units",
            })
            pos_id += 1
        if pos_id >= target_size:
            break

    while pos_id < target_size:
        iid, spec = fixed_bond(pos_id)
        iid_unique = f"{iid}-P{pos_id}"
        spec_copy = copy.deepcopy(spec)
        spec_copy["spec"]["id"] = iid_unique
        desc = _instrument_desc(spec_copy)
        positions.append({
            "position_id": f"POS-{pos_id} ({desc})",
            "entity_id": f"FUND-{(pos_id % 3) + 1}",
            "instrument_id": iid_unique,
            "instrument_spec": spec_copy,
            "quantity": 1.0,
            "unit": "units",
        })
        pos_id += 1

    return positions


def _fixup_structured_ids(spec: dict, old_prefix: str, new_id: str):
    if spec.get("type") != "structured_credit":
        return
    inner = spec["spec"]
    inner["pool"]["id"] = f"{new_id}-POOL"
    for j, tr in enumerate(inner.get("tranches", {}).get("tranches", [])):
        tr["id"] = f"{new_id}-T{j}"
    for j, asset in enumerate(inner.get("pool", {}).get("assets", [])):
        asset["id"] = f"{new_id}-A{j}"


print("Position builder ready.")

Position builder ready.


In [6]:
def make_portfolio_json(portfolio_id: str, num_positions: int) -> str:
    positions = build_positions(num_positions)
    entity_ids = sorted({p["entity_id"] for p in positions})
    return json.dumps({
        "id": portfolio_id,
        "as_of": AS_OF_STR,
        "base_ccy": "USD",
        "entities": {eid: {"id": eid} for eid in entity_ids},
        "positions": positions,
    })


base_portfolio_json = make_portfolio_json("MULTI-ASSET-BASE", 40)
base_obj = json.loads(base_portfolio_json)
print(f"Base portfolio: {len(base_obj['positions'])} positions, {len(base_obj['entities'])} entities")

type_counts: dict[str, int] = {}
for p in base_obj["positions"]:
    t = p["instrument_spec"]["type"]
    type_counts[t] = type_counts.get(t, 0) + 1
for t, c in sorted(type_counts.items()):
    print(f"  {t:25s}: {c}")

Base portfolio: 40 positions, 3 entities
  bond                     : 13
  cds_index                : 1
  cds_option               : 1
  cds_tranche              : 1
  convertible_bond         : 3
  credit_default_swap      : 4
  equity                   : 2
  fx_swap                  : 1
  interest_rate_future     : 2
  interest_rate_swap       : 4
  revolving_credit         : 1
  structured_credit        : 2
  swaption                 : 2
  term_loan                : 2
  variance_swap            : 1


## 5. Value the base portfolio

In [7]:
t0 = time.perf_counter()
val_result = value_portfolio(base_portfolio_json, MARKET_JSON)
elapsed_base = time.perf_counter() - t0

val_obj = json.loads(val_result)
total_pv = float(val_obj["total_base_ccy"]["amount"])
n_priced = len(val_obj["position_values"])
print(f"Base portfolio PV: {total_pv:,.2f} USD  ({n_priced} positions valued in {elapsed_base:.3f}s)")

Base portfolio PV: 77,423,552.99 USD  (40 positions valued in 0.074s)


### Per-position breakdown

In [8]:
for pos_id, pv_info in list(val_obj["position_values"].items())[:12]:
    pv = float(pv_info["value_base"]["amount"])
    print(f"  {pos_id:<36s} PV = {pv:>18,.2f} USD")
if n_priced > 12:
    print(f"  ... ({n_priced - 12} more positions)")

  POS-0 (Fixed 3Y 4.0%)                PV =       1,000,882.76 USD
  POS-1 (Fixed 4Y 4.5%)                PV =       1,019,115.84 USD
  POS-2 (Fixed 5Y 5.0%)                PV =       1,045,846.13 USD
  POS-3 (Fixed 6Y 5.5%)                PV =       1,076,979.51 USD
  POS-4 (FRN 2Y SOFR+100bp)            PV =       1,033,262.50 USD
  POS-5 (FRN 3Y SOFR+125bp)            PV =       1,053,744.58 USD
  POS-6 (FRN 4Y SOFR+150bp)            PV =       1,080,659.29 USD
  POS-7 (TL Fixed 3Y 500bp)            PV =       9,143,636.58 USD
  POS-8 (TL Fixed 4Y 550bp)            PV =       9,327,540.51 USD
  POS-9 (Revolver 2Y 10/50M)           PV =      15,766,129.37 USD
  POS-10 (CDS Pay 3Y 80bp)             PV =         150,450.62 USD
  POS-11 (CDS Rec 4Y 100bp)            PV =        -148,415.24 USD
  ... (28 more positions)


### Per-position risk sensitivities

`value_portfolio` automatically computes the standard portfolio metric set for every position:
**DV01**, **bucketed DV01** (key-rate), **CS01**, **bucketed CS01**, **delta**, **gamma**, **vega**, **rho**, **pv01**, and **theta**. Not every instrument supports every metric -- the engine silently omits inapplicable ones.

In [9]:
RISK_KEYS = ["dv01", "cs01", "theta", "delta", "gamma", "vega", "rho", "pv01"]

header = f"{'Position':<36s} {'Type':<24s}" + "".join(f"{k:>12s}" for k in RISK_KEYS)
print(header)
print("-" * len(header))

for pid, pval in val_obj["position_values"].items():
    vr = pval.get("valuation_result")
    if not vr:
        continue
    measures = vr.get("measures", {})
    itype = pval.get("instrument_type", "")
    if not itype:
        for p in base_obj["positions"]:
            if p["position_id"] == pid:
                itype = p["instrument_spec"]["type"]
                break
    row = f"{pid:<36s} {itype:<24s}"
    for k in RISK_KEYS:
        v = measures.get(k)
        row += f"{v:>12.2f}" if v is not None else f"{'--':>12s}"
    print(row)

Position                             Type                            dv01        cs01       theta       delta       gamma        vega         rho        pv01
-------------------------------------------------------------------------------------------------------------------------------------------------------------
POS-0 (Fixed 3Y 4.0%)                bond                         -280.04     -285.85      119.85          --          --          --          --          --
POS-1 (Fixed 4Y 4.5%)                bond                         -369.45     -378.08      121.78          --          --          --          --          --
POS-2 (Fixed 5Y 5.0%)                bond                         -458.88     -470.72      124.74          --          --          --          --          --
POS-3 (Fixed 6Y 5.5%)                bond                         -547.58     -563.00      128.24          --          --          --          --          --
POS-4 (FRN 2Y SOFR+100bp)            bond           

### Bucketed DV01 -- rate risk by tenor

Key-rate DV01 decomposes the parallel DV01 across standard tenor buckets for every curve the position depends on. The keys follow the pattern `bucketed_dv01::<curve_id>::<tenor>`.

In [10]:
def extract_bucketed(measures: dict, prefix: str) -> dict[str, float]:
    """Extract bucketed metric entries matching 'prefix::curve::tenor'."""
    buckets: dict[str, float] = {}
    for k, v in measures.items():
        if k.startswith(prefix + "::") and k.count("::") == 2:
            buckets[k] = v
    return buckets


def pivot_buckets(val_obj: dict, base_obj: dict, prefix: str) -> tuple[list[str], dict]:
    """Build {(position, type): {tenor_key: value}} and collect all tenor keys."""
    all_keys: set[str] = set()
    rows = {}
    for pid, pval in val_obj["position_values"].items():
        vr = pval.get("valuation_result")
        if not vr:
            continue
        buckets = extract_bucketed(vr.get("measures", {}), prefix)
        if not buckets:
            continue
        itype = ""
        for p in base_obj["positions"]:
            if p["position_id"] == pid:
                itype = p["instrument_spec"]["type"]
                break
        rows[(pid, itype)] = buckets
        all_keys.update(buckets.keys())
    sorted_keys = sorted(all_keys, key=lambda k: k.split("::")[-1])
    return sorted_keys, rows


def tenor_sort_key(full_key: str) -> tuple[str, float]:
    """Sort by curve then numeric tenor."""
    parts = full_key.split("::")
    curve = parts[1] if len(parts) > 1 else ""
    tenor_str = parts[-1]
    multiplier = {"m": 1/12, "y": 1}.get(tenor_str[-1], 1) if tenor_str else 0
    try:
        num = float(tenor_str[:-1]) * multiplier
    except (ValueError, IndexError):
        num = 999.0
    return (curve, num)


sorted_keys, dv01_rows = pivot_buckets(val_obj, base_obj, "bucketed_dv01")
sorted_keys.sort(key=tenor_sort_key)

short_labels = [k.split("::", 1)[-1] for k in sorted_keys]
header = f"{'Position':<36s} {'Type':<20s}" + "".join(f"{lbl:>18s}" for lbl in short_labels)
print("Bucketed DV01 (USD/bp)")
print(header)
print("-" * len(header))
for (pid, itype), bkts in dv01_rows.items():
    row = f"{pid:<36s} {itype:<20s}"
    for k in sorted_keys:
        v = bkts.get(k)
        row += f"{v:>18.4f}" if v is not None else f"{'':>18s}"
    print(row)

Bucketed DV01 (USD/bp)
Position                             Type                       EUR-OIS::3m       EUR-OIS::6m       EUR-OIS::1y       EUR-OIS::2y       EUR-OIS::3y       EUR-OIS::5y       EUR-OIS::7y      EUR-OIS::10y      EUR-OIS::15y      EUR-OIS::20y      EUR-OIS::30y        USD-IG::3m        USD-IG::6m        USD-IG::1y        USD-IG::2y        USD-IG::3y        USD-IG::5y        USD-IG::7y       USD-IG::10y       USD-IG::15y       USD-IG::20y       USD-IG::30y       USD-OIS::3m       USD-OIS::6m       USD-OIS::1y       USD-OIS::2y       USD-OIS::3y       USD-OIS::5y       USD-OIS::7y      USD-OIS::10y      USD-OIS::15y      USD-OIS::20y      USD-OIS::30y   USD-SOFR-3M::3m   USD-SOFR-3M::6m   USD-SOFR-3M::1y   USD-SOFR-3M::2y   USD-SOFR-3M::3y   USD-SOFR-3M::5y   USD-SOFR-3M::7y  USD-SOFR-3M::10y  USD-SOFR-3M::15y  USD-SOFR-3M::20y  USD-SOFR-3M::30y
-------------------------------------------------------------------------------------------------------------------------------

### Bucketed CS01 -- credit spread risk by tenor

Analogous to DV01 but bumping hazard/credit curves. Keys follow `bucketed_cs01::<curve_id>::<tenor>`. Bond CS01 uses z-spread bump and keys off the instrument ID.

In [11]:
sorted_cs_keys, cs01_rows = pivot_buckets(val_obj, base_obj, "bucketed_cs01")
sorted_cs_keys.sort(key=tenor_sort_key)

if cs01_rows:
    short_cs = [k.split("::", 1)[-1] for k in sorted_cs_keys]
    header = f"{'Position':<36s} {'Type':<20s}" + "".join(f"{lbl:>18s}" for lbl in short_cs)
    print("Bucketed CS01 (USD/bp)")
    print(header)
    print("-" * len(header))
    for (pid, itype), bkts in cs01_rows.items():
        row = f"{pid:<36s} {itype:<20s}"
        for k in sorted_cs_keys:
            v = bkts.get(k)
            row += f"{v:>18.4f}" if v is not None else f"{'':>18s}"
        print(row)
else:
    print("No bucketed CS01 data (no credit instruments in base portfolio).")

Bucketed CS01 (USD/bp)
Position                             Type                       CDX-HAZ::3m       CDX-HAZ::6m       CDX-HAZ::1y       CDX-HAZ::2y       CDX-HAZ::3y       CDX-HAZ::5y       CDX-HAZ::7y      CDX-HAZ::10y      CDX-HAZ::15y      CDX-HAZ::20y      CDX-HAZ::30y   CORP-HAZARD::3m   CORP-HAZARD::6m   CORP-HAZARD::1y   CORP-HAZARD::2y   CORP-HAZARD::3y   CORP-HAZARD::5y   CORP-HAZARD::7y  CORP-HAZARD::10y  CORP-HAZARD::15y  CORP-HAZARD::20y  CORP-HAZARD::30y
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
POS-10 (CDS Pay 3Y 80bp)             credit_default_swap             

## 6. Portfolio-level risk report

`aggregate_metrics` sums all summable sensitivities across positions (with FX conversion when needed) and provides breakdowns by entity.

In [12]:
agg_json = aggregate_metrics(val_result, "USD", MARKET_JSON, AS_OF_STR)
agg = json.loads(agg_json)

print("Portfolio Risk Summary (base portfolio, 40 positions)")
print("=" * 60)

headline_metrics = ["dv01", "cs01", "theta", "delta", "gamma", "vega", "rho", "pv01"]
for m in headline_metrics:
    info = agg["aggregated"].get(m)
    if info:
        print(f"  {m:<12s}  {info['total']:>18,.2f} USD")
    else:
        print(f"  {m:<12s}  {'n/a':>18s}")

Portfolio Risk Summary (base portfolio, 40 positions)
  dv01                  -14,815.03 USD
  cs01                   14,540.56 USD
  theta                  12,669.01 USD
  delta              79,203,138.82 USD
  gamma             181,044,462.22 USD
  vega                      877.64 USD
  rho                     4,151.40 USD
  pv01                   -1,726.76 USD


### Aggregated bucketed DV01 by tenor

Portfolio-wide rate exposure summed across all positions.

In [13]:
def print_agg_buckets(agg: dict, prefix: str, title: str):
    """Print aggregated bucketed metrics with entity breakdown."""
    bucket_keys = sorted(
        (k for k in agg["aggregated"] if k.startswith(prefix + "::")),
        key=tenor_sort_key,
    )
    if not bucket_keys:
        print(f"No {title} data in aggregation.")
        return

    entities = sorted({e for info in agg["aggregated"].values() for e in info.get("by_entity", {})})
    header = f"{'Bucket':<30s} {'Total':>14s}" + "".join(f"{e:>14s}" for e in entities)
    print(title)
    print(header)
    print("-" * len(header))
    grand_total = 0.0
    for k in bucket_keys:
        info = agg["aggregated"][k]
        label = k.split("::", 1)[-1]
        total = info["total"]
        grand_total += total
        row = f"{label:<30s} {total:>14,.2f}"
        for e in entities:
            row += f"{info['by_entity'].get(e, 0):>14,.2f}"
        print(row)
    print("-" * len(header))
    print(f"{'TOTAL':<30s} {grand_total:>14,.2f}")
    print()


print_agg_buckets(agg, "bucketed_dv01", "Aggregated Bucketed DV01 (USD/bp)")

Aggregated Bucketed DV01 (USD/bp)
Bucket                                  Total        FUND-1        FUND-2        FUND-3
---------------------------------------------------------------------------------------
EUR-OIS::3m                             -1.27         -1.27          0.00          0.00
EUR-OIS::6m                             53.91         53.91          0.00          0.00
EUR-OIS::1y                              0.10          0.10          0.00          0.00
EUR-OIS::2y                             -0.00         -0.00          0.00          0.00
EUR-OIS::3y                              0.00          0.00          0.00          0.00
EUR-OIS::5y                              0.00          0.00          0.00          0.00
EUR-OIS::7y                              0.00          0.00          0.00          0.00
EUR-OIS::10y                             0.00          0.00          0.00          0.00
EUR-OIS::15y                             0.00          0.00          0.00          0.0

### Aggregated bucketed CS01 by tenor

Portfolio-wide credit spread exposure across hazard curves.

In [14]:
print_agg_buckets(agg, "bucketed_cs01", "Aggregated Bucketed CS01 (USD/bp)")

Aggregated Bucketed CS01 (USD/bp)
Bucket                                  Total        FUND-1        FUND-2        FUND-3
---------------------------------------------------------------------------------------
CDX-HAZ::3m                          7,473.29      6,908.43          0.00        564.86
CDX-HAZ::6m                          7,473.29      6,908.43          0.00        564.86
CDX-HAZ::1y                          7,473.29      6,908.43          0.00        564.86
CDX-HAZ::2y                          7,473.29      6,908.43          0.00        564.86
CDX-HAZ::3y                         10,816.49      9,778.65          0.00      1,037.84
CDX-HAZ::5y                          7,649.60      6,746.91          0.00        902.69
CDX-HAZ::7y                          7,649.60      6,746.91          0.00        902.69
CDX-HAZ::10y                             0.00          0.00          0.00          0.00
CDX-HAZ::15y                             0.00          0.00          0.00          0.0

### Risk by entity

Break down headline sensitivities by fund entity to see risk concentration.

In [15]:
entities = sorted({e for info in agg["aggregated"].values() for e in info.get("by_entity", {})})
header = f"{'Metric':<12s} {'Total':>16s}" + "".join(f"{e:>16s}" for e in entities)
print("Risk by Entity (USD)")
print(header)
print("-" * len(header))
for m in headline_metrics:
    info = agg["aggregated"].get(m)
    if not info:
        continue
    row = f"{m:<12s} {info['total']:>16,.2f}"
    for e in entities:
        row += f"{info['by_entity'].get(e, 0):>16,.2f}"
    print(row)

Risk by Entity (USD)
Metric                  Total          FUND-1          FUND-2          FUND-3
-----------------------------------------------------------------------------
dv01               -14,815.03       -1,754.73       -6,413.52       -6,646.79
cs01                14,540.56       20,146.60         -941.07       -4,664.96
theta               12,669.01        3,385.28        4,135.49        5,148.24
delta           79,203,138.82     -450,557.16   41,293,471.35   38,360,224.63
gamma          181,044,462.22  154,550,823.25   19,841,780.02    6,651,858.95
vega                   877.64          468.96            5.97          402.71
rho                  4,151.40          -52.13           -0.00        4,203.53
pv01                -1,726.76        4,485.39       -2,585.14       -3,627.02


## 7. Scale to 500 positions

In [16]:
portfolio_500_json = make_portfolio_json("MULTI-ASSET-500", 500)
obj_500 = json.loads(portfolio_500_json)

type_counts_500: dict[str, int] = {}
for p in obj_500["positions"]:
    t = p["instrument_spec"]["type"]
    type_counts_500[t] = type_counts_500.get(t, 0) + 1
print("500-position portfolio instrument mix:")
for t, c in sorted(type_counts_500.items()):
    print(f"  {t:25s}: {c:4d}  ({100*c/500:.0f}%)")

t0 = time.perf_counter()
val_500 = value_portfolio(portfolio_500_json, MARKET_JSON)
elapsed_500 = time.perf_counter() - t0

v500 = json.loads(val_500)
total_500 = float(v500["total_base_ccy"]["amount"])
n_500 = len(v500["position_values"])
print(f"\nTotal PV: {total_500:,.2f} USD")
print(f"Valued {n_500} positions in {elapsed_500:.3f}s  ({n_500/elapsed_500:,.0f} positions/sec)")

500-position portfolio instrument mix:
  bond                     :   90  (18%)
  cds_index                :   20  (4%)
  cds_option               :   20  (4%)
  cds_tranche              :   20  (4%)
  convertible_bond         :   45  (9%)
  credit_default_swap      :   50  (10%)
  equity                   :   25  (5%)
  fx_swap                  :   20  (4%)
  interest_rate_future     :   25  (5%)
  interest_rate_swap       :   50  (10%)
  revolving_credit         :   20  (4%)
  structured_credit        :   40  (8%)
  swaption                 :   25  (5%)
  term_loan                :   30  (6%)
  variance_swap            :   20  (4%)

Total PV: 1,219,609,614.47 USD
Valued 500 positions in 0.193s  (2,594 positions/sec)


## 8. Scale to 3,000 positions

In [17]:
portfolio_3000_json = make_portfolio_json("MULTI-ASSET-3000", 3000)
obj_3000 = json.loads(portfolio_3000_json)

type_counts_3k: dict[str, int] = {}
for p in obj_3000["positions"]:
    t = p["instrument_spec"]["type"]
    type_counts_3k[t] = type_counts_3k.get(t, 0) + 1
print("3,000-position portfolio instrument mix:")
for t, c in sorted(type_counts_3k.items()):
    print(f"  {t:25s}: {c:4d}  ({100*c/3000:.0f}%)")

t0 = time.perf_counter()
val_3000 = value_portfolio(portfolio_3000_json, MARKET_JSON)
elapsed_3000 = time.perf_counter() - t0

v3000 = json.loads(val_3000)
total_3000 = float(v3000["total_base_ccy"]["amount"])
n_3000 = len(v3000["position_values"])
print(f"\nTotal PV: {total_3000:,.2f} USD")
print(f"Valued {n_3000} positions in {elapsed_3000:.3f}s  ({n_3000/elapsed_3000:,.0f} positions/sec)")

3,000-position portfolio instrument mix:
  bond                     :  540  (18%)
  cds_index                :  120  (4%)
  cds_option               :  120  (4%)
  cds_tranche              :  120  (4%)
  convertible_bond         :  270  (9%)
  credit_default_swap      :  300  (10%)
  equity                   :  150  (5%)
  fx_swap                  :  120  (4%)
  interest_rate_future     :  150  (5%)
  interest_rate_swap       :  300  (10%)
  revolving_credit         :  120  (4%)
  structured_credit        :  240  (8%)
  swaption                 :  150  (5%)
  term_loan                :  180  (6%)
  variance_swap            :  120  (4%)

Total PV: 7,307,146,951.98 USD
Valued 3000 positions in 0.955s  (3,142 positions/sec)


## 9. Scale to 25,000 positions

In [18]:
portfolio_25k_json = make_portfolio_json("MULTI-ASSET-25K", 25000)
obj_25k = json.loads(portfolio_25k_json)

type_counts_25k: dict[str, int] = {}
for p in obj_25k["positions"]:
    t = p["instrument_spec"]["type"]
    type_counts_25k[t] = type_counts_25k.get(t, 0) + 1
print("25,000-position portfolio instrument mix:")
for t, c in sorted(type_counts_25k.items()):
    print(f"  {t:25s}: {c:5d}  ({100*c/25000:.0f}%)")

t0 = time.perf_counter()
val_25k = value_portfolio(portfolio_25k_json, MARKET_JSON)
elapsed_25k = time.perf_counter() - t0

v25k = json.loads(val_25k)
total_25k = float(v25k["total_base_ccy"]["amount"])
n_25k = len(v25k["position_values"])
print(f"\nTotal PV: {total_25k:,.2f} USD")
print(f"Valued {n_25k} positions in {elapsed_25k:.3f}s  ({n_25k/elapsed_25k:,.0f} positions/sec)")

25,000-position portfolio instrument mix:
  bond                     :  4500  (18%)
  cds_index                :  1000  (4%)
  cds_option               :  1000  (4%)
  cds_tranche              :  1000  (4%)
  convertible_bond         :  2250  (9%)
  credit_default_swap      :  2500  (10%)
  equity                   :  1250  (5%)
  fx_swap                  :  1000  (4%)
  interest_rate_future     :  1250  (5%)
  interest_rate_swap       :  2500  (10%)
  revolving_credit         :  1000  (4%)
  structured_credit        :  2000  (8%)
  swaption                 :  1250  (5%)
  term_loan                :  1500  (6%)
  variance_swap            :  1000  (4%)

Total PV: 60,896,396,756.79 USD
Valued 25000 positions in 9.295s  (2,690 positions/sec)


## 10. Performance summary

In [19]:
print("Portfolio valuation throughput")
print("=" * 75)
for label, n, elapsed, pv in [
    ("Base (40)", n_priced, elapsed_base, total_pv),
    ("Medium (500)", n_500, elapsed_500, total_500),
    ("Large (3,000)", n_3000, elapsed_3000, total_3000),
    ("XL (25,000)", n_25k, elapsed_25k, total_25k),
]:
    rate = n / elapsed if elapsed > 0 else float("inf")
    print(f"  {label:20s}  {n:6d} positions  {elapsed:7.3f}s  {rate:>8,.0f} pos/s  PV={pv:>18,.2f} USD")

Portfolio valuation throughput
  Base (40)                 40 positions    0.074s       540 pos/s  PV=     77,423,552.99 USD
  Medium (500)             500 positions    0.193s     2,594 pos/s  PV=  1,219,609,614.47 USD
  Large (3,000)           3000 positions    0.955s     3,142 pos/s  PV=  7,307,146,951.98 USD
  XL (25,000)            25000 positions    9.295s     2,690 pos/s  PV= 60,896,396,756.79 USD


## 11. 1-day P&L attribution

Simulate an overnight market move (rates +5 bp, credit spreads +10 bp, equity -2%, vol +1 pt, FX EUR/USD -0.5%) and attribute the portfolio P&L across risk factors using `attribute_pnl` with the **Parallel** methodology.

Each position is attributed independently, then results are aggregated to portfolio level.

In [20]:
AS_OF_T1 = "2025-01-16"

mc_t1 = MarketContext()

mc_t1.insert(DiscountCurve(
    "USD-OIS", date(2025, 1, 16),
    [(0.0, 1.0), (0.25, 0.9885), (0.5, 0.9770), (1.0, 0.9540), (2.0, 0.9085),
     (3.0, 0.8680), (5.0, 0.7975), (7.0, 0.7270), (10.0, 0.6465)],
    day_count="act_365f",
))

mc_t1.insert(DiscountCurve(
    "EUR-OIS", date(2025, 1, 16),
    [(0.0, 1.0), (0.25, 0.9922), (0.5, 0.9845), (1.0, 0.9690), (2.0, 0.9385),
     (3.0, 0.9080), (5.0, 0.8475), (7.0, 0.7870), (10.0, 0.6965)],
    day_count="act_365f",
))

mc_t1.insert(ForwardCurve(
    "USD-SOFR-3M", 0.25,
    [(0.0, 0.0455), (0.5, 0.0465), (1.0, 0.0475), (2.0, 0.0485),
     (3.0, 0.0495), (5.0, 0.0505), (10.0, 0.0525)],
    base_date=date(2025, 1, 16),
    day_count="act_360",
))

mc_t1.insert(DiscountCurve(
    "USD-IG", date(2025, 1, 16),
    [(0.0, 1.0), (1.0, 0.9435), (3.0, 0.8375), (5.0, 0.7465), (10.0, 0.5755)],
    day_count="act_365f",
))

mc_t1.insert(DiscountCurve(
    "USD-CREDIT-BBB", date(2025, 1, 16),
    [(0.0, 1.0), (1.0, 0.9330), (3.0, 0.8165), (5.0, 0.7155), (10.0, 0.5145)],
    day_count="act_365f",
))

mc_t1.insert(HazardCurve(
    "CORP-HAZARD", date(2025, 1, 16),
    [(0.5, 0.019), (1.0, 0.021), (3.0, 0.025), (5.0, 0.029), (10.0, 0.033)],
    recovery_rate=0.40,
))

mc_t1.insert(HazardCurve(
    "CDX-HAZ", date(2025, 1, 16),
    [(1.0, 0.011), (3.0, 0.016), (5.0, 0.021), (10.0, 0.026)],
    recovery_rate=0.40,
))

state_t1 = json.loads(mc_t1.to_json())

state_t1["curves"].append({
    "type": "base_correlation",
    "id": "CDX-BC",
    "detachment_points": [3.0, 7.0, 10.0, 15.0, 30.0, 100.0],
    "correlations": [0.18, 0.28, 0.35, 0.45, 0.65, 0.85],
})

state_t1["credit_indices"] = [{
    "id": "CDX.NA.IG.HAZARD",
    "num_constituents": 125,
    "recovery_rate": 0.4,
    "index_credit_curve_id": "CDX-HAZ",
    "base_correlation_curve_id": "CDX-BC",
}]

state_t1["surfaces"] = [
    {"id": "CDS-SPREAD-VOL", "expiries": [0.25, 0.5, 1.0, 2.0],
     "strikes": [50.0, 100.0, 150.0, 200.0, 300.0],
     "secondary_axis": "strike", "interpolation_mode": "vol",
     "vols_row_major": [0.31] * 20},
    {"id": "TECH-VOL", "expiries": [0.25, 0.5, 1.0, 2.0, 5.0],
     "strikes": [20.0, 30.0, 40.0, 50.0, 60.0],
     "secondary_axis": "strike", "interpolation_mode": "vol",
     "vols_row_major": [0.36] * 25},
    {"id": "USD-SWPNVOL", "expiries": [0.25, 0.5, 1.0, 2.0, 5.0],
     "strikes": [0.02, 0.03, 0.04, 0.05, 0.06],
     "secondary_axis": "strike", "interpolation_mode": "vol",
     "vols_row_major": [0.21] * 25},
]

state_t1.setdefault("prices", {})
state_t1["prices"]["TECH"] = {"price": {"amount": 41.16, "currency": "USD"}}
state_t1["prices"]["AAPL-SPOT"] = {"price": {"amount": 181.30, "currency": "USD"}}
state_t1["prices"]["AAPL-DIV"] = {"unitless": 0.005}
state_t1["prices"]["SPX-SPOT"] = {"price": {"amount": 5096.0, "currency": "USD"}}
state_t1["prices"]["SPX-DIV"] = {"unitless": 0.015}

state_t1["fx"] = {
    "config": {"pivot_currency": "USD", "enable_triangulation": False, "cache_capacity": 256},
    "quotes": [["EUR", "USD", 1.0746]],
}

MARKET_T1_JSON = json.dumps(state_t1)

print("T1 market snapshot ready (rates +5bp, credit +10bp, equity -2%, vol +1pt, EUR/USD -0.5%)")

T1 market snapshot ready (rates +5bp, credit +10bp, equity -2%, vol +1pt, EUR/USD -0.5%)


In [21]:
FACTOR_COLS = [
    ("carry", "Carry"),
    ("rates_curves_pnl", "Rates"),
    ("credit_curves_pnl", "Credit"),
    ("vol_pnl", "Vol"),
    ("fx_pnl", "FX"),
    ("market_scalars_pnl", "Scalars"),
    ("cross_factor_pnl", "Cross"),
    ("residual", "Residual"),
]


def run_portfolio_attribution(portfolio_obj: dict, label: str) -> dict:
    """Attribute every position and aggregate to portfolio level."""
    t_start = time.perf_counter()
    totals = {col: 0.0 for _, col in FACTOR_COLS}
    totals["total_pnl"] = 0.0
    position_attrs = []
    errors = 0

    for pos in portfolio_obj["positions"]:
        inst_json = json.dumps(pos["instrument_spec"])
        try:
            attr = attribute_pnl(
                inst_json, MARKET_JSON, MARKET_T1_JSON,
                AS_OF_STR, AS_OF_T1, "Parallel",
            )
            row = {"position_id": pos["position_id"], "type": pos["instrument_spec"]["type"]}
            row["total_pnl"] = attr.total_pnl
            totals["total_pnl"] += attr.total_pnl
            for prop, col in FACTOR_COLS:
                v = getattr(attr, prop)
                row[col] = v
                totals[col] += v
            position_attrs.append(row)
        except Exception:
            errors += 1

    elapsed = time.perf_counter() - t_start
    n = len(portfolio_obj["positions"])
    print(f"{label}: {n} positions attributed in {elapsed:.3f}s "
          f"({n / elapsed:,.0f} pos/s, {errors} errors)")
    return {"totals": totals, "positions": position_attrs, "elapsed": elapsed}


base_attr = run_portfolio_attribution(base_obj, "Base (40)")

Base (40): 40 positions attributed in 0.031s (1,292 pos/s, 1 errors)


### Base portfolio P&L decomposition

In [22]:
def print_attribution_summary(result: dict, title: str):
    totals = result["totals"]
    print(f"\n{title}")
    print("=" * 70)
    print(f"  {'Total P&L':<16s}  {totals['total_pnl']:>18,.2f} USD")
    print(f"  {'-' * 52}")
    for _, col in FACTOR_COLS:
        print(f"  {col:<16s}  {totals[col]:>18,.2f} USD")
    explained = sum(totals[col] for _, col in FACTOR_COLS if col != "Residual")
    print(f"  {'-' * 52}")
    print(f"  {'Explained':<16s}  {explained:>18,.2f} USD")
    if abs(totals["total_pnl"]) > 1e-6:
        pct = totals["Residual"] / totals["total_pnl"] * 100
        print(f"  {'Residual %':<16s}  {pct:>17.2f}%")


print_attribution_summary(base_attr, "Base Portfolio Attribution (40 positions)")


Base Portfolio Attribution (40 positions)
  Total P&L                 -71,321.05 USD
  ----------------------------------------------------
  Carry                      11,242.74 USD
  Rates                    -122,113.27 USD
  Credit                     39,748.65 USD
  Vol                           470.69 USD
  FX                            -83.01 USD
  Scalars                      -740.00 USD
  Cross                        -149.99 USD
  Residual                      303.14 USD
  ----------------------------------------------------
  Explained                 -71,624.19 USD
  Residual %                    -0.43%


### Per-position attribution detail (base portfolio)

In [23]:
cols = ["Total"] + [col for _, col in FACTOR_COLS]
header = f"{'Position':<36s} {'Type':<22s}" + "".join(f"{c:>12s}" for c in cols)
print(header)
print("-" * len(header))

for row in base_attr["positions"]:
    line = f"{row['position_id']:<36s} {row['type']:<22s}"
    line += f"{row['total_pnl']:>12,.0f}"
    for _, col in FACTOR_COLS:
        line += f"{row[col]:>12,.0f}"
    print(line)

Position                             Type                         Total       Carry       Rates      Credit         Vol          FX     Scalars       Cross    Residual
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
POS-0 (Fixed 3Y 4.0%)                bond                        -2,041         120      -2,161           0           0           0           0           0           0
POS-1 (Fixed 4Y 4.5%)                bond                        -2,454         122      -2,576           0           0           0           0           0           0
POS-2 (Fixed 5Y 5.0%)                bond                        -2,811         125      -2,936           0           0           0           0           0           0
POS-3 (Fixed 6Y 5.5%)                bond                        -3,254         128      -3,382           0           0           0           0           0     

### Attribution by instrument type

In [24]:
type_agg: dict[str, dict[str, float]] = {}
for row in base_attr["positions"]:
    t = row["type"]
    if t not in type_agg:
        type_agg[t] = {"total_pnl": 0.0}
        for _, col in FACTOR_COLS:
            type_agg[t][col] = 0.0
    type_agg[t]["total_pnl"] += row["total_pnl"]
    for _, col in FACTOR_COLS:
        type_agg[t][col] += row[col]

header = f"{'Instrument Type':<26s}" + "".join(f"{c:>12s}" for c in cols)
print(header)
print("-" * len(header))
for t in sorted(type_agg):
    line = f"{t:<26s}"
    line += f"{type_agg[t]['total_pnl']:>12,.0f}"
    for _, col in FACTOR_COLS:
        line += f"{type_agg[t][col]:>12,.0f}"
    print(line)

Instrument Type                  Total       Carry       Rates      Credit         Vol          FX     Scalars       Cross    Residual
--------------------------------------------------------------------------------------------------------------------------------------
bond                           -35,200       1,642     -36,843           0           0           0           0           0           0
cds_index                       24,267        -137        -441      24,800           0           0           0         -44          88
cds_option                      22,908          51        -687      23,490           4           0           0         -47          97
convertible_bond               -12,370         432     -12,803           0           0           0           0          -0           0
credit_default_swap             -8,536           9          22      -8,542           0           0           0          25         -51
equity                            -740           0     

### Attribution at scale: 500, 3,000, and 25,000 positions

In [25]:
attr_500 = run_portfolio_attribution(obj_500, "Medium (500)")
print_attribution_summary(attr_500, "500-Position Portfolio Attribution")

Medium (500): 500 positions attributed in 0.504s (993 pos/s, 20 errors)

500-Position Portfolio Attribution
  Total P&L              -1,314,558.86 USD
  ----------------------------------------------------
  Carry                     168,699.63 USD
  Rates                  -1,732,061.23 USD
  Credit                    187,997.79 USD
  Vol                        67,391.71 USD
  FX                         -3,322.82 USD
  Scalars                    -9,250.00 USD
  Cross                      -4,429.12 USD
  Residual                   10,415.16 USD
  ----------------------------------------------------
  Explained              -1,324,974.03 USD
  Residual %                    -0.79%


In [26]:
attr_3000 = run_portfolio_attribution(obj_3000, "Large (3,000)")
print_attribution_summary(attr_3000, "3,000-Position Portfolio Attribution")

Large (3,000): 3000 positions attributed in 3.011s (996 pos/s, 120 errors)

3,000-Position Portfolio Attribution
  Total P&L              -8,074,207.06 USD
  ----------------------------------------------------
  Carry                   1,012,606.72 USD
  Rates                 -10,606,129.37 USD
  Credit                  1,149,168.09 USD
  Vol                       408,994.48 USD
  FX                        -19,936.89 USD
  Scalars                   -55,500.00 USD
  Cross                     -27,248.32 USD
  Residual                   63,838.23 USD
  ----------------------------------------------------
  Explained              -8,138,045.29 USD
  Residual %                    -0.79%


In [27]:
attr_25k = run_portfolio_attribution(obj_25k, "XL (25,000)")
print_attribution_summary(attr_25k, "25,000-Position Portfolio Attribution")

XL (25,000): 25000 positions attributed in 21.860s (1,144 pos/s, 1000 errors)

25,000-Position Portfolio Attribution
  Total P&L             -67,204,881.52 USD
  ----------------------------------------------------
  Carry                   8,436,915.54 USD
  Rates                 -88,334,869.57 USD
  Credit                  9,580,405.49 USD
  Vol                     3,434,538.32 USD
  FX                       -166,140.76 USD
  Scalars                  -462,500.00 USD
  Cross                    -228,922.84 USD
  Residual                  535,692.30 USD
  ----------------------------------------------------
  Explained             -67,740,573.82 USD
  Residual %                    -0.80%


### Attribution throughput summary

In [28]:
print("P&L Attribution throughput (Parallel method)")
print("=" * 70)
for label, result, n in [
    ("Base (40)", base_attr, 40),
    ("Medium (500)", attr_500, 500),
    ("Large (3,000)", attr_3000, 3000),
    ("XL (25,000)", attr_25k, 25000),
]:
    elapsed = result["elapsed"]
    total = result["totals"]["total_pnl"]
    rate = n / elapsed if elapsed > 0 else float("inf")
    print(f"  {label:20s}  {n:5d} positions  {elapsed:7.3f}s  "
          f"{rate:>6,.0f} pos/s  P&L={total:>18,.2f} USD")

P&L Attribution throughput (Parallel method)
  Base (40)                40 positions    0.031s   1,292 pos/s  P&L=        -71,321.05 USD
  Medium (500)            500 positions    0.504s     993 pos/s  P&L=     -1,314,558.86 USD
  Large (3,000)          3000 positions    3.011s     996 pos/s  P&L=     -8,074,207.06 USD
  XL (25,000)           25000 positions   21.860s   1,144 pos/s  P&L=    -67,204,881.52 USD


## 12. Historical & stress scenarios

Apply three scenarios to the portfolio and compare the P&L impact:

| Scenario | Description |
|---|---|
| **GFC 2008** | Rates -200bp (bull steepener), IG credit +150bp, HY credit +400bp, equity -50%, vol +200% |
| **COVID 2020** | Rates -150bp, IG credit +100bp, HY credit +300bp, equity -34%, vol +250% |
| **Static: Rates +200bp / Credit widen** | Parallel rate rise +200bp, IG credit +100bp, HY credit +500bp |

Each scenario is built as a `ScenarioSpec`, applied to the base market via `apply_scenario_to_market`, then the portfolio is revalued under the stressed market.

In [29]:
gfc_ops = json.dumps([
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": -200.0},
    {"kind": "curve_node_bp", "curve_kind": "discount", "curve_id": "USD-OIS",
     "nodes": [["2Y", -50.0], ["5Y", 0.0], ["10Y", 50.0]], "match_mode": "interpolate"},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "EUR-OIS", "bp": -150.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-IG", "bp": 100.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-CREDIT-BBB", "bp": 600.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD",
     "discount_curve_id": "USD-OIS", "bp": 400.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CDX-HAZ",
     "discount_curve_id": "USD-OIS", "bp": 150.0},
    {"kind": "equity_price_pct", "ids": ["TECH", "AAPL-SPOT", "SPX-SPOT"], "pct": -50.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "equity", "surface_id": "TECH-VOL", "pct": 200.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "credit", "surface_id": "CDS-SPREAD-VOL", "pct": 150.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "swaption", "surface_id": "USD-SWPNVOL", "pct": 100.0},
    {"kind": "market_fx_pct", "base": "EUR", "quote": "USD", "pct": -10.0},
])
gfc_spec = build_scenario_spec(
    "gfc_2008", gfc_ops,
    name="Global Financial Crisis 2008",
    description="Lehman collapse: rates rally, credit blowout, equity crash, vol spike",
)

covid_ops = json.dumps([
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": -150.0},
    {"kind": "curve_node_bp", "curve_kind": "discount", "curve_id": "USD-OIS",
     "nodes": [["2Y", -30.0], ["10Y", 10.0]], "match_mode": "interpolate"},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "EUR-OIS", "bp": -100.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-IG", "bp": 50.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-CREDIT-BBB", "bp": 450.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD",
     "discount_curve_id": "USD-OIS", "bp": 300.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CDX-HAZ",
     "discount_curve_id": "USD-OIS", "bp": 100.0},
    {"kind": "equity_price_pct", "ids": ["TECH", "AAPL-SPOT", "SPX-SPOT"], "pct": -34.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "equity", "surface_id": "TECH-VOL", "pct": 250.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "credit", "surface_id": "CDS-SPREAD-VOL", "pct": 200.0},
    {"kind": "vol_surface_parallel_pct", "surface_kind": "swaption", "surface_id": "USD-SWPNVOL", "pct": 120.0},
    {"kind": "market_fx_pct", "base": "EUR", "quote": "USD", "pct": -5.0},
])
covid_spec = build_scenario_spec(
    "covid_2020", covid_ops,
    name="COVID 2020 Market Shock",
    description="Pandemic panic: emergency easing, credit widening, equity crash, vol spike",
)

static_ops = json.dumps([
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": 200.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "EUR-OIS", "bp": 200.0},
    {"kind": "curve_parallel_bp", "curve_kind": "forward", "curve_id": "USD-SOFR-3M", "bp": 200.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-IG", "bp": 300.0},
    {"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-CREDIT-BBB", "bp": 700.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD",
     "discount_curve_id": "USD-OIS", "bp": 500.0},
    {"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CDX-HAZ",
     "discount_curve_id": "USD-OIS", "bp": 100.0},
])
static_spec = build_scenario_spec(
    "rates_up_credit_wide", static_ops,
    name="Rates +200bp / Credit Widen (IG +100, HY +500)",
    description="Static stress: parallel rate rise with IG and HY credit spread widening",
)

SCENARIOS = [
    ("GFC 2008", gfc_spec),
    ("COVID 2020", covid_spec),
    ("Rates +200 / Credit Wide", static_spec),
]

print(f"Built {len(SCENARIOS)} scenario specs")

Built 3 scenario specs


### Scenario helper: apply, revalue, and compute P&L

In [30]:
def run_scenario(scenario_name: str, scenario_spec: str, portfolio_json: str,
                  base_val_obj: dict, num_positions: int) -> dict:
    """Apply scenario to market, revalue portfolio, and compute P&L vs base."""
    t_start = time.perf_counter()

    result = apply_scenario_to_market(scenario_spec, MARKET_JSON, AS_OF_STR)
    stressed_market = result["market_json"]
    ops_applied = result["operations_applied"]
    warnings = result["warnings"]

    stressed_val = value_portfolio(portfolio_json, stressed_market)
    stressed_obj = json.loads(stressed_val)

    base_pv = sum(
        float(v["value_base"]["amount"]) for v in base_val_obj["position_values"].values()
    )
    stressed_pv = sum(
        float(v["value_base"]["amount"]) for v in stressed_obj["position_values"].values()
    )
    total_pnl = stressed_pv - base_pv

    position_pnls = {}
    for pos_id in base_val_obj["position_values"]:
        b = float(base_val_obj["position_values"][pos_id]["value_base"]["amount"])
        s_pos = stressed_obj["position_values"].get(pos_id)
        s = float(s_pos["value_base"]["amount"]) if s_pos else b
        position_pnls[pos_id] = s - b

    elapsed = time.perf_counter() - t_start
    return {
        "name": scenario_name,
        "ops_applied": ops_applied,
        "warnings": warnings,
        "base_pv": base_pv,
        "stressed_pv": stressed_pv,
        "total_pnl": total_pnl,
        "pnl_pct": total_pnl / abs(base_pv) * 100 if abs(base_pv) > 1e-6 else 0.0,
        "position_pnls": position_pnls,
        "elapsed": elapsed,
        "num_positions": num_positions,
    }

### Base portfolio (40 positions) under each scenario

In [31]:
base_40_results = []
for name, spec in SCENARIOS:
    r = run_scenario(name, spec, base_portfolio_json, val_obj, 40)
    base_40_results.append(r)
    print(f"{name:30s}  ops={r['ops_applied']:2d}  "
          f"P&L={r['total_pnl']:>16,.0f} USD  ({r['pnl_pct']:+.1f}%)  "
          f"{r['elapsed']:.3f}s  warnings={len(r['warnings'])}")

GFC 2008                        ops=14  P&L=       6,483,267 USD  (+8.4%)  0.058s  warnings=0
COVID 2020                      ops=14  P&L=       4,810,700 USD  (+6.2%)  0.065s  warnings=0
Rates +200 / Credit Wide        ops= 7  P&L=         -57,739 USD  (-0.1%)  0.061s  warnings=0


### Per-position scenario P&L (base portfolio, top movers)

In [32]:
scenario_names = [r["name"] for r in base_40_results]
header = f"{'Position':<14s}" + "".join(f"{n:>22s}" for n in scenario_names)
print("Top 10 P&L movers (sorted by worst GFC loss)")
print(header)
print("-" * len(header))

all_pos_ids = list(base_40_results[0]["position_pnls"].keys())
sorted_pos = sorted(all_pos_ids, key=lambda p: base_40_results[0]["position_pnls"].get(p, 0))

for pos_id in sorted_pos[:10]:
    line = f"{pos_id:<14s}"
    for r in base_40_results:
        pnl = r["position_pnls"].get(pos_id, 0)
        line += f"{pnl:>22,.0f}"
    print(line)

Top 10 P&L movers (sorted by worst GFC loss)
Position                    GFC 2008            COVID 2020Rates +200 / Credit Wide
----------------------------------------------------------------------------------
POS-13 (CDS Rec 6Y 140bp)            -1,079,582              -821,837            -1,180,482
POS-11 (CDS Rec 4Y 100bp)              -785,511              -594,786              -880,647
POS-33 (CB TECH 5Y 2.5%)              -142,556              -112,363              -161,230
POS-32 (CB TECH 4Y 2.0%)              -124,608               -97,352              -141,633
POS-31 (CB TECH 3Y 1.5%)              -111,255               -86,645              -126,797
POS-27 (AAPL 100sh)                -9,250                -6,290                     0
POS-28 (AAPL 100sh)                -9,250                -6,290                     0
POS-21 (IRS Pay 5Y 5.0%)                -4,674                -3,354               854,521
POS-20 (IRS Receive 4Y 4.5%)                -3,615                -2,

### Scale scenarios to 500, 3,000, and 25,000 positions

In [33]:
all_scenario_results = {"40": base_40_results}

for label, n, port_json, bval in [
    ("500", 500, portfolio_500_json, v500),
    ("3000", 3000, portfolio_3000_json, v3000),
    ("25000", 25000, portfolio_25k_json, v25k),
]:
    results = []
    for name, spec in SCENARIOS:
        r = run_scenario(name, spec, port_json, bval, n)
        results.append(r)
    all_scenario_results[label] = results

    print(f"\n{'=' * 80}")
    print(f"  {n:,d}-position portfolio")
    print(f"{'=' * 80}")
    for r in results:
        print(f"  {r['name']:30s}  P&L={r['total_pnl']:>18,.0f} USD  "
              f"({r['pnl_pct']:+.1f}%)  {r['elapsed']:.3f}s")


  500-position portfolio
  GFC 2008                        P&L=       106,555,109 USD  (+8.7%)  0.221s
  COVID 2020                      P&L=        76,928,993 USD  (+6.3%)  0.239s
  Rates +200 / Credit Wide        P&L=         6,972,737 USD  (+0.6%)  0.220s

  3,000-position portfolio
  GFC 2008                        P&L=       637,764,013 USD  (+8.7%)  1.313s
  COVID 2020                      P&L=       459,442,226 USD  (+6.3%)  1.269s
  Rates +200 / Credit Wide        P&L=        31,977,398 USD  (+0.4%)  1.354s

  25,000-position portfolio
  GFC 2008                        P&L=     5,315,803,941 USD  (+8.7%)  10.054s
  COVID 2020                      P&L=     3,830,016,528 USD  (+6.3%)  11.708s
  Rates +200 / Credit Wide        P&L=       269,762,195 USD  (+0.4%)  11.366s


### Scenario summary across all portfolio sizes

In [34]:
print("Scenario P&L impact across portfolio sizes")
print("=" * 100)
header = f"{'Scenario':30s}  {'Positions':>9s}  {'Base PV':>18s}  {'Stressed PV':>18s}  {'P&L':>18s}  {'%':>7s}  {'Time':>7s}"
print(header)
print("-" * 100)

for size_label in ["40", "500", "3000", "25000"]:
    for r in all_scenario_results[size_label]:
        print(f"{r['name']:30s}  {r['num_positions']:>9,d}  "
              f"{r['base_pv']:>18,.0f}  {r['stressed_pv']:>18,.0f}  "
              f"{r['total_pnl']:>18,.0f}  {r['pnl_pct']:>+6.1f}%  {r['elapsed']:>6.2f}s")
    print()

Scenario P&L impact across portfolio sizes
Scenario                        Positions             Base PV         Stressed PV                 P&L        %     Time
----------------------------------------------------------------------------------------------------
GFC 2008                               40          77,423,553          83,906,820           6,483,267    +8.4%    0.06s
COVID 2020                             40          77,423,553          82,234,253           4,810,700    +6.2%    0.06s
Rates +200 / Credit Wide               40          77,423,553          77,365,814             -57,739    -0.1%    0.06s

GFC 2008                              500       1,219,609,614       1,326,164,724         106,555,109    +8.7%    0.22s
COVID 2020                            500       1,219,609,614       1,296,538,608          76,928,993    +6.3%    0.24s
Rates +200 / Credit Wide              500       1,219,609,614       1,226,582,352           6,972,737    +0.6%    0.22s

GFC 2008      

## Takeaways

- **Seventeen instrument types** share a single `MarketContext` snapshot and a single `value_portfolio` call.
- Every position automatically receives **bucketed DV01**, **bucketed CS01**, **delta**, **gamma**, **vega**, **rho**, **pv01**, and **theta** where applicable -- the standard risk set an investment group monitors.
- `aggregate_metrics` consolidates position-level sensitivities into portfolio totals and entity breakdowns, handling FX conversion for multi-currency books.
- The JSON-based portfolio spec scales linearly: position factory functions parameterize maturity, coupon, strike, and side to generate diverse but valid instruments.
- **1-day P&L attribution** decomposes overnight MTM changes into carry, rates, credit, vol, FX, and cross-factor contributions with a small residual, running at scale across the full book.
- **Historical and stress scenarios** (GFC, COVID, rates+credit) apply parameterised shocks via `apply_scenario_to_market` and revalue the entire portfolio in a single call, producing scenario P&L at all three scales.
- At 25,000 positions the Rust engine handles valuation, attribution, and scenario analysis in seconds, making full-book risk reporting practical even at institutional scale.
- Structured credit (CLO/ABS) instruments are the heaviest per-position; the throughput numbers are blended averages across all types.